# DPO Pipeline

In [1]:
# Environment Setup
import os
os.environ["CC"] = os.path.expanduser("~/gcc_wrapper")
os.environ.pop("GCC_EXEC_PREFIX", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:

# Imports
import datetime
from datasets import load_dataset, Dataset, DatasetDict
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig


In [3]:

# Data Pipeline Function
def create_clean_dpo_pipeline():
    """
    Create a clean DPO training pipeline with proper splits and filtering
    """
    
    # 1. Load and combine datasets
    print("Loading datasets...")
    
    # Load your M1 dataset (smaller, domain-specific)
    m1_dpo = load_dataset("ciacco/m1_dpo_filtered_highest_total", split="train")
    print(f"M1 dataset: {len(m1_dpo)} samples")
    
    # Load juicy mix (larger, general)
    juicy_mix = load_dataset("ciacco/m1_dpo_juicy_mix", split="train")
    print(f"Juicy mix dataset: {len(juicy_mix)} samples")
    
    # 2. Standardize M1 dataset format (add missing fields and fix MCQ prompts)
    def standardize_m1_sample(sample):
        # Add question options to MCQ prompts if missing
        prompt = sample['prompt']
        if sample.get('question_type') == 'mcq' and sample.get('question_options'):
            if "Options:" not in prompt and sample['question_options']:
                prompt = f"{prompt}\n\nOptions:\n{sample['question_options']}"
        
        return {
            'prompt': prompt,
            'chosen': sample['chosen'],
            'rejected': sample['rejected'],
            'source': 'm1_dataset'
        }
    
    # Apply standardization to M1 dataset
    m1_standardized = m1_dpo.map(standardize_m1_sample)
    m1_standardized = m1_standardized.select_columns(['prompt', 'chosen', 'rejected', 'source'])
    
    # 4. Downsample juicy mix to manageable size
    juicy_sample_size = min(10000, len(juicy_mix))  # Sample 10k from juicy mix
    juicy_sampled = juicy_mix.shuffle(seed=42).select(range(juicy_sample_size))
    
    # Ensure juicy mix has source column
    if 'source' not in juicy_sampled.column_names:
        juicy_sampled = juicy_sampled.add_column('source', ['juicy_mix'] * len(juicy_sampled))
    
    print(f"Sampled juicy mix: {len(juicy_sampled)} samples")
    
    # 5. Combine datasets
    # Option 1: M1 only (for domain-specific training)
    m1_only = m1_standardized
    
    # Option 2: Combined dataset with balanced sampling
    # Take all M1 + sampled juicy mix
    combined_list = list(m1_standardized) + list(juicy_sampled)
    combined_dataset = Dataset.from_list(combined_list)
    
    print(f"Combined dataset: {len(combined_dataset)} samples")
    
    # 6. Create train/eval splits for both datasets
    def create_splits(dataset, test_size=0.1, seed=42):
        """Create train/eval splits"""
        dataset = dataset.shuffle(seed=seed)
        split_idx = int(len(dataset) * (1 - test_size))
        
        train_dataset = dataset.select(range(split_idx))
        eval_dataset = dataset.select(range(split_idx, len(dataset)))
        
        return DatasetDict({
            'train': train_dataset,
            'eval': eval_dataset
        })
    
    # Create splits
    m1_splits = create_splits(m1_only)
    combined_splits = create_splits(combined_dataset)
    
    print(f"M1 splits - Train: {len(m1_splits['train'])}, Eval: {len(m1_splits['eval'])}")
    print(f"Combined splits - Train: {len(combined_splits['train'])}, Eval: {len(combined_splits['eval'])}")
    
    # 7. Push to hub
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    
    # Push M1 only splits
    m1_splits.push_to_hub(f"ciacco/m1_dpo_clean_splits_{timestamp}", private=True)
    
    # Push combined splits  
    combined_splits.push_to_hub(f"ciacco/m1_dpo_combined_splits_{timestamp}", private=True)
    
    return m1_splits, combined_splits, timestamp


In [4]:

# Run Data Pipeline
m1_splits, combined_splits, timestamp = create_clean_dpo_pipeline()


Loading datasets...
M1 dataset: 7584 samples
Juicy mix dataset: 40630 samples


Map:   0%|          | 0/7584 [00:00<?, ? examples/s]

Sampled juicy mix: 10000 samples
Combined dataset: 17584 samples
M1 splits - Train: 6825, Eval: 759
Combined splits - Train: 15825, Eval: 1759


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


In [5]:

# Load Model and Tokenizer
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base",     
                                          torch_dtype="auto",
                                          device_map="auto", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B-Base",
                                            torch_dtype="auto",
                                            device_map="auto", trust_remote_code=True)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model memory footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"CUDA available: {torch.cuda.is_available()}")


Loading model and tokenizer...
Model memory footprint: 1.19 GB
CUDA available: True


In [6]:

# Setup Training Configuration
# Choose dataset: 'm1_only' or 'combined'
dataset_choice = 'combined' #'m1_only'  # Change this to 'combined' if you want both datasets

if dataset_choice == 'm1_only':
    train_dataset = m1_splits['train']
    eval_dataset = m1_splits['eval']
    experiment_name = "m1_clean_only"
else:
    train_dataset = combined_splits['train']
    eval_dataset = combined_splits['eval']
    experiment_name = "m1_combined_clean"

print(f"Using {dataset_choice} dataset")
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

# Create experiment name
run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
run_name = f"{experiment_name}_{run_timestamp}"
output_dir = f"outputs/dpo_{run_name}"

print(f"Run name: {run_name}")
print(f"Output directory: {output_dir}")


Using combined dataset
Train dataset: 15825 samples
Eval dataset: 1759 samples
Run name: m1_combined_clean_20250608_1531
Output directory: outputs/dpo_m1_combined_clean_20250608_1531


In [10]:

# Create DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-7,
        logging_steps=25,
        eval_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=200,
        save_total_limit=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=output_dir,
        run_name=run_name,
        beta=0.1,
        report_to="tensorboard",
        max_length=2048,
        max_prompt_length=1024,
        gradient_checkpointing=True,
        dataloader_drop_last=True,
        remove_unused_columns=False,
    ),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)


Extracting prompt in train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:

# Train Model
print(f"Starting training: {run_name}")
print("=" * 50)
dpo_trainer.train()


Starting training: m1_combined_clean_20250608_1531


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
100,0.679200,0.696834,0.002768,0.003606,0.286526,-0.000837,-377.831665,-322.903564,-1.305897,-1.093270
200,0.696700,0.697869,-0.000051,0.001914,0.291643,-0.001967,-377.859802,-322.920471,-1.305713,-1.093148
300,0.699900,0.694220,0.001525,-0.003653,0.302445,0.005178,-377.844055,-322.976135,-1.305758,-1.093047
400,0.694400,0.691686,0.007642,-0.002259,0.301876,0.009904,-377.782928,-322.962189,-1.304996,-1.092781
500,0.693100,0.692015,0.010461,0.000517,0.316089,0.009950,-377.754761,-322.934448,-1.303283,-1.091069
600,0.685900,0.691983,0.012075,0.001250,0.317226,0.010829,-377.738678,-322.927124,-1.302198,-1.090286
700,0.692500,0.688904,0.014334,-0.000943,0.328596,0.015284,-377.716064,-322.949036,-1.300774,-1.088575
800,0.670500,0.687389,0.019580,0.001167,0.342240,0.018416,-377.663666,-322.927948,-1.300545,-1.088745
900,0.687000,0.685968,0.022282,0.001257,0.338829,0.021034,-377.636688,-322.927063,-1.299848,-1.087616
1000,0.693400,0.687527,0.021674,0.002642,0.358158,0.019035,-377.642761,-322.913208,-1.298497,-1.086370


In [14]:
# Load model and tokenizer directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_ckpt = AutoTokenizer.from_pretrained("/home/project-m3-2025-intelligent-agents/code/train_dpo/outputs/dpo_m1_combined_clean_20250608_1531/checkpoint-3956",     
                                          torch_dtype="auto",
                                          device_map="auto")
model_ckpt = AutoModelForCausalLM.from_pretrained("/home/project-m3-2025-intelligent-agents/code/train_dpo/outputs/dpo_m1_combined_clean_20250608_1531/checkpoint-3956",
                                            torch_dtype="auto",
                                            device_map="auto")

In [15]:
model_ckpt.push_to_hub("ciacco/dpo_m1_combined_clean_20250608_1531", private=True)
tokenizer_ckpt.push_to_hub("ciacco/dpo_m1_combined_clean_20250608_1531", private=True)

Uploading...:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ciacco/dpo_m1_combined_clean_20250608_1531/commit/826dd299a48537307bc3968905164181e0838c66', commit_message='Upload tokenizer', commit_description='', oid='826dd299a48537307bc3968905164181e0838c66', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ciacco/dpo_m1_combined_clean_20250608_1531', endpoint='https://huggingface.co', repo_type='model', repo_id='ciacco/dpo_m1_combined_clean_20250608_1531'), pr_revision=None, pr_num=None)

In [ ]:
# Save and Upload Model
print("Training completed! Saving model...")

# Save final model
final_model_name = f"ciacco/MNLP_M3_dpo_{run_name}"
model.push_to_hub(final_model_name, private=True)
tokenizer.push_to_hub(final_model_name, private=True)

print(f"Model saved as: {final_model_name}")
print(f"Training logs available in: {output_dir}")
print("Check TensorBoard logs with: tensorboard --logdir outputs/")



## Recommended Workflow:

1. **Create the new notebook**: `Clean_DPO_Pipeline.ipynb`
2. **Run cells 1-4**: This creates and uploads the clean datasets
3. **Run cells 5-6**: Load model and choose dataset configuration
4. **Run cell 7**: Create trainer (check configuration first)
5. **Run cell 8**: Start training
6. **Run cell 9**: Save the final model

## Benefits of this approach:

- **Clean separation**: Keep your old experiments intact while having a clean new pipeline
- **Modular**: Each cell has a specific purpose
- **Reproducible**: Timestamped runs and consistent naming
- **Flexible**: Easy to switch between M1-only and combined datasets
- **Proper evaluation**: Built-in train/eval splits and monitoring

This gives you a fresh start with all the improvements while keeping your previous work as reference.